# Big Data with sparklyr

## What you'll learn
- What Apache Spark is and how sparklyr connects R to it
- How to start a local Spark session
- How to read data into Spark and work with it using familiar dplyr verbs
- How to bring results back to R with `collect()`
- When sparklyr is the right choice

## Prerequisites
- Completed Notebooks 01 and 02
- Generated the large dataset: `python scripts/generate_large_data.py`
- The `codingcs` environment is activated (it includes Java and Spark)

## What Is Spark?

Apache Spark is an engine for processing large datasets. It was designed to work across **clusters** of computers — many machines working together. But it also works on a single machine, using multiple CPU cores in parallel.

**sparklyr** is the R package that lets you talk to Spark using the dplyr syntax you already know. You don't need to learn a new language — you use `filter()`, `group_by()`, and `summarize()` just like before. The difference is that Spark does the heavy lifting behind the scenes.

Think of it this way: dplyr is one person doing calculations. Spark is a team of people working in parallel on the same problem.

## Connecting to Spark

Before you can use Spark, you need to start a **Spark session** — think of it as turning on the Spark engine. We'll use `master = "local"` which means Spark runs on your own computer (no cluster needed).

In [ ]:
library(sparklyr)
library(dplyr)

In [ ]:
# Start a local Spark session
# This may take 10-20 seconds the first time
sc <- spark_connect(master = "local", version = "3.5.3")

If this is your first time, sparklyr may need to download and install Spark. This is a one-time setup and may take a few minutes.

The variable `sc` is your **Spark connection** — you'll pass it to every function that needs to talk to Spark.

## Reading Data into Spark

With regular R, you use `read_csv()` to load data into a data frame. With sparklyr, you use `spark_read_csv()` to load data into a **Spark DataFrame**.

The data doesn't live in R's memory — it lives inside Spark. R just holds a reference to it.

In [ ]:
# Read the large sales dataset into Spark
sales_spark <- spark_read_csv(
  sc,
  name = "sales",
  path = "../data/sales_large.csv",
  infer_schema = TRUE
)

In [ ]:
# Peek at the data (only shows first few rows)
sales_spark

Notice the output shows the data source is Spark (`table<sales>` with `Database: spark_connection`) — this tells you the data lives in Spark, not in R's memory.

## Exploring Data in Spark

You can check the shape and structure of your Spark DataFrame with these functions:

In [ ]:
# Number of rows
sdf_nrow(sales_spark)

In [ ]:
# Column names and types
sdf_schema(sales_spark)

In [ ]:
# Column names only
colnames(sales_spark)

## Using dplyr on Spark

Here's the magic of sparklyr: **you use the same dplyr verbs you already know**. The only difference is that the operations run inside Spark instead of in R.

### select() — Pick Columns

In [ ]:
sales_spark |>
  select(date, product, category, unit_price)

### filter() — Pick Rows

In [ ]:
# Electronics only
sales_spark |>
  filter(category == "Electronics")

### mutate() — Add Columns

In [ ]:
# Calculate total price (quantity * unit_price)
sales_spark |>
  mutate(total_price = quantity * unit_price) |>
  select(product, quantity, unit_price, total_price)

### group_by() + summarize() — Aggregate

In [ ]:
# Total revenue by category
sales_spark |>
  mutate(total_price = quantity * unit_price) |>
  group_by(category) |>
  summarize(
    total_revenue = sum(total_price, na.rm = TRUE),
    num_transactions = n()
  ) |>
  arrange(desc(total_revenue))

### arrange() — Sort

In [ ]:
# Top 10 most expensive transactions
sales_spark |>
  mutate(total_price = quantity * unit_price) |>
  arrange(desc(total_price)) |>
  head(10)

## collect() — Bring Results Back to R

All the operations above run inside Spark. The results stay in Spark too. If you want to bring results back into R (as a regular data frame) so you can plot them or do further analysis with base R, use `collect()`.

**Important:** Only collect **small** results (summaries, filtered subsets). Collecting the entire 100K-row dataset defeats the purpose of using Spark.

In [ ]:
# Summarize in Spark, then collect the small result into R
revenue_by_region <- sales_spark |>
  mutate(total_price = quantity * unit_price) |>
  group_by(region) |>
  summarize(total_revenue = sum(total_price, na.rm = TRUE)) |>
  collect()

# Now revenue_by_region is a regular R data frame
class(revenue_by_region)

In [ ]:
revenue_by_region

## Seeing the SQL Behind the Scenes

Spark actually translates your dplyr code into SQL and runs that. You can see the generated SQL with `show_query()`. This is useful for learning SQL and for debugging.

In [ ]:
sales_spark |>
  filter(category == "Electronics") |>
  group_by(product) |>
  summarize(avg_price = mean(unit_price, na.rm = TRUE)) |>
  arrange(desc(avg_price)) |>
  show_query()

## Timing Comparison

Let's compare how long a grouped aggregation takes in base R vs Spark.

In [ ]:
library(readr)

# Read into R for comparison
sales_r <- read_csv("../data/sales_large.csv", show_col_types = FALSE)

In [ ]:
# Time the dplyr (in-memory) version
system.time({
  sales_r |>
    mutate(total_price = quantity * unit_price) |>
    group_by(category, region) |>
    summarize(revenue = sum(total_price), .groups = "drop")
})

In [ ]:
# Time the Spark version
system.time({
  sales_spark |>
    mutate(total_price = quantity * unit_price) |>
    group_by(category, region) |>
    summarize(revenue = sum(total_price, na.rm = TRUE)) |>
    collect()
})

With 100K rows, base R is likely faster (Spark has startup overhead). The benefit of Spark shows up with millions of rows or when data is distributed across a cluster.

## Disconnecting

Always disconnect from Spark when you're done. This frees up resources.

In [ ]:
spark_disconnect(sc)

---
## Summary

- **sparklyr** connects R to Apache Spark — a powerful engine for large-scale data processing
- You write **dplyr code** and Spark executes it, potentially across many cores or machines
- Use `spark_read_csv()` to load data, dplyr verbs to transform it, and `collect()` to bring small results back to R
- Spark shines with very large datasets or cluster deployments; for smaller data, it has more overhead than base R

**Next up:** [04 - Big Data with Arrow](04-big-data-arrow.ipynb) — a faster, lighter alternative for single-machine big data work.